# Pembersihan Data Mentah

Membersihkan 8 sumber data mentah di `data/raw/` menjadi deret waktu harian, masing-masing disimpan sebagai file tersendiri di `data/processed/`.

In [6]:
import pandas as pd
import numpy as np
import os

RAW = os.path.join('..', 'data', 'raw')
PROCESSED = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED, exist_ok=True)


## 1. Harga Emas Antam

Sumber: `antam_historical_gold_prices_download_asli/Antam_historical_gold_prices.csv` (di luar sub-folder `raw/` bawaan unduhan) karena sudah menyertakan kolom `Date` hasil konversi dari timestamp.

**Langkah pembersihan:**
1. Ambil pengumuman **terakhir** pada tiap hari kalender (bisa lebih dari satu pengumuman per hari).
2. Reindex ke hari kerja (Senin-Jumat) - Antam tidak mengumumkan di akhir pekan.
3. Hari kerja tanpa pengumuman (libur nasional, mis. Idul Fitri, Natal/Tahun Baru) diisi **forward-fill**: harga pengumuman terakhir tetap berlaku. (Protokol v9: sebelumnya interpolasi linear; dibuang karena memakai harga pengumuman *berikutnya* = lookahead, dan menghasilkan harga yang tidak pernah diumumkan.)

In [7]:
gold_path = os.path.join(RAW, 'antam_historical_gold_prices_download_asli', 'Antam_historical_gold_prices.csv')
gold_raw = pd.read_csv(gold_path, parse_dates=['Date']).sort_values('Date')
gold_raw['tanggal'] = gold_raw['Date'].dt.normalize()

harian = gold_raw.groupby('tanggal')['Gold Price'].last()
harian.index = pd.to_datetime(harian.index)

hari_kerja = pd.bdate_range(harian.index.min(), harian.index.max())
# Harga yang berlaku = pengumuman TERAKHIR sampai hari itu (forward-fill, kausal). Pengumuman
# di akhir pekan ikut dihitung (union index); hari kerja tanpa pengumuman memakai harga sebelumnya.
harga_emas = harian.reindex(harian.index.union(hari_kerja)).ffill().reindex(hari_kerja)
assert harga_emas.notna().all(), 'ada hari kerja tanpa harga'
n_tanpa_pengumuman = int((~hari_kerja.isin(harian.index)).sum())
harga_emas.index.name = 'tanggal'
harga_emas.name = 'harga_jual'

harga_emas.to_frame().to_csv(os.path.join(PROCESSED, 'harga_emas_bersih.csv'))
print(f'Pengumuman mentah -> hari kerja bersih : {len(gold_raw)} -> {len(harga_emas)}')
print(f'Hari kerja tanpa pengumuman (harga sebelumnya dipertahankan): {n_tanpa_pengumuman}')
print(f'Periode: {harga_emas.index.min().date()} s.d. {harga_emas.index.max().date()}')
harga_emas.head()


Pengumuman mentah -> hari kerja bersih : 5144 -> 4326
Hari kerja tanpa pengumuman (harga sebelumnya dipertahankan): 198
Periode: 2010-01-04 s.d. 2026-08-03


tanggal
2010-01-04    408000.0
2010-01-05    410000.0
2010-01-06    410000.0
2010-01-07    412000.0
2010-01-08    410000.0
Freq: B, Name: harga_jual, dtype: float64

## 2. BI-Rate (unduhan baru dari BIS)

File unduhan BIS punya beberapa baris metadata sebelum data sebenarnya dimulai, dengan dua kolom nilai: frekuensi harian (`D.ID`) dan bulanan (`M.ID`). Yang dipakai kolom **harian** sedikit lebih presisi, karena tanggal persis perubahan suku bunga ikut tertangkap.

In [8]:
bis = pd.read_csv(
    os.path.join(RAW, 'bis_bi_rate_export.csv'),
    skiprows=9, encoding='utf-8-sig',
    names=['tanggal', 'bi_rate_daily', 'bi_rate_monthly'],
)
bis['tanggal'] = pd.to_datetime(bis['tanggal'])
bi_rate = bis.set_index('tanggal')['bi_rate_daily'].dropna()
bi_rate = bi_rate[~bi_rate.index.duplicated(keep='last')].sort_index()
bi_rate.name = 'bi_rate'

bi_rate.to_frame().to_csv(os.path.join(PROCESSED, 'bi_rate_bersih.csv'))
print(f'Observasi BI-Rate harian: {len(bi_rate)}')
print(f'Periode: {bi_rate.index.min().date()} s.d. {bi_rate.index.max().date()}')
bi_rate.tail()


Observasi BI-Rate harian: 5175
Periode: 2005-07-05 s.d. 2026-09-03


tanggal
2026-08-28    5.75
2026-08-31    5.75
2026-09-01    5.75
2026-09-02    5.75
2026-09-03    5.75
Name: bi_rate, dtype: float64

## 3. Indikator Makro Lain

Pembersihan: menstandardkan nama index, membuang duplikat tanggal, membuang baris kosong, dan **membuang titik data rusak (glitch)** pada empat deret harga/indeks.

In [9]:
ATURAN_GLITCH = {              # (ambang rasio vs median 7 hari, pakai aturan lonjakan-balik?)
    'usd_idr': (1.08, True), 'dxy': (1.08, True),      # kurs: ketat
    'sp500': (2.0, False), 'ihsg': (2.0, False),       # indeks saham: hanya kesalahan besar
}


def buang_glitch(s, ambang, lonjakan, maks_putaran=3):
    """Return (deret_bersih, titik_dibuang). Deviasi di skala log terhadap median 7 hari (titik tengah)."""
    s = s.copy()
    dibuang = []
    for _ in range(maks_putaran):
        ls = np.log(s)
        med = ls.rolling(7, center=True, min_periods=4).median()
        flag = (ls - med).abs() > np.log(ambang)
        if lonjakan:                                    # naik/turun >3% lalu balik hampir penuh
            r, r_next = ls.diff(), ls.diff().shift(-1)
            besar = pd.concat([r.abs(), r_next.abs()], axis=1).max(axis=1)
            flag |= ((r.abs() > 0.03) & (r_next.abs() > 0.03) & (np.sign(r) != np.sign(r_next))
                     & ((r + r_next).abs() < 0.4 * besar))
        if not flag.any():
            break
        dibuang.append(s[flag])
        s = s[~flag]
    return s, (pd.concat(dibuang).sort_index() if dibuang else s.iloc[0:0])


hasil, glitch_log = {}, []
for nama in ['dxy', 'sp500', 'usd_idr', 'ihsg', 'us10yr', 'ffr']:
    s = pd.read_csv(os.path.join(RAW, f'{nama}.csv'), index_col=0, parse_dates=True).iloc[:, 0]
    s.index = s.index.normalize()
    s.index.name = 'tanggal'
    s = s[~s.index.duplicated(keep='last')].sort_index().dropna()
    s.name = nama

    if nama in ATURAN_GLITCH:
        s, dibuang = buang_glitch(s, *ATURAN_GLITCH[nama])
        for tgl, v in dibuang.items():
            sebelum = s.loc[:tgl]
            glitch_log.append({'indikator': nama, 'tanggal': tgl.date(), 'nilai_dibuang': float(v),
                               'nilai_valid_sebelumnya': float(sebelum.iloc[-1]) if len(sebelum) else np.nan})
    assert (s > 0).all(), f'{nama}: ada nilai <= 0 setelah pembersihan'

    s.to_frame().to_csv(os.path.join(PROCESSED, f'{nama}_bersih.csv'))
    hasil[nama] = s
    print(f'{nama:8s}: {len(s):5d} baris | {s.index.min().date()} s.d. {s.index.max().date()}')

glitch_df = pd.DataFrame(glitch_log, columns=['indikator', 'tanggal', 'nilai_dibuang', 'nilai_valid_sebelumnya'])
glitch_df.to_csv(os.path.join(PROCESSED, 'glitch_dibuang.csv'), index=False)
print(f'\nTitik yang dibuang per indikator:\n{glitch_df.indikator.value_counts().to_string() if len(glitch_df) else "tidak ada"}')

# pemeriksaan kewajaran: pergerakan harian terbesar sesudah pembersihan
print('\n|log-return| harian terbesar sesudah pembersihan:')
for nama in ['usd_idr', 'dxy', 'sp500', 'ihsg']:
    lr = np.log(hasil[nama]).diff().abs()
    print(f'  {nama:8s}: {lr.max() * 100:5.1f}% pada {lr.idxmax().date()}')
if len(glitch_df):
    display(glitch_df)

dxy     :  4340 baris | 2009-06-01 s.d. 2026-08-28
sp500   :  4339 baris | 2009-06-01 s.d. 2026-08-28
usd_idr :  4466 baris | 2009-06-01 s.d. 2026-08-28
ihsg    :  4180 baris | 2009-06-01 s.d. 2026-08-28
us10yr  :  4346 baris | 2009-06-01 s.d. 2026-10-14
ffr     :   208 baris | 2009-06-01 s.d. 2026-09-01

Titik yang dibuang per indikator:
indikator
usd_idr    23

|log-return| harian terbesar sesudah pembersihan:
  usd_idr :   4.6% pada 2020-03-24
  dxy     :   2.4% pada 2015-12-03
  sp500   :  12.8% pada 2020-03-16
  ihsg    :   9.7% pada 2020-03-26


,indikator,tanggal,nilai_dibuang,nilai_valid_sebelumnya
0,usd_idr,2010-03-05,8898.400391,9328.900391
1,usd_idr,2010-11-01,888.109985,8923.799805
2,usd_idr,2011-10-05,8622.000000,8897.450195
3,usd_idr,2012-02-07,892.000000,8928.099609
4,usd_idr,2013-07-15,9612.450195,9968.000000
5,usd_idr,2013-07-22,9612.450195,10032.000000
6,usd_idr,2013-10-14,10787.000000,11463.000000
7,usd_idr,2013-10-21,10758.000000,11303.000000
8,usd_idr,2013-10-28,9612.450195,11128.000000
9,usd_idr,2013-11-11,9612.450195,11363.000000


## Ringkasan

8 file bersih sudah tersimpan di `data/processed/`.

In [10]:
import glob
for f in sorted(glob.glob(os.path.join(PROCESSED, '*_bersih.csv'))):
    print(os.path.basename(f))


bi_rate_bersih.csv
dxy_bersih.csv
ffr_bersih.csv
harga_emas_bersih.csv
ihsg_bersih.csv
sp500_bersih.csv
us10yr_bersih.csv
usd_idr_bersih.csv
